<a href="https://colab.research.google.com/github/MuhammadRayan07/Weather-Analysis/blob/main/notebooks/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weather Data Analysis and Prediction
## Notebook 1: Data Collection

**Purpose:** Collect daily historical weather data for 20 cities (2016-2025) from the Open-Meteo Historical Weather API.
**Data source:** Open-Meteo (https://open-meteo.com), reanalysis-based data.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path("/content/drive/MyDrive/weather-analysis")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Folders ready:", RAW.exists(), PROCESSED.exists())

Mounted at /content/drive
Folders ready: True True


In [2]:
import pandas as pd

cities = pd.DataFrame([
    ("Lahore","Pakistan",31.5497,74.3436),("Karachi","Pakistan",24.8607,67.0011),
    ("Dubai","UAE",25.2048,55.2708),("Delhi","India",28.6139,77.2090),
    ("Tokyo","Japan",35.6762,139.6503),("Singapore","Singapore",1.3521,103.8198),
    ("Beijing","China",39.9042,116.4074),("Sydney","Australia",-33.8688,151.2093),
    ("Cairo","Egypt",30.0444,31.2357),("Nairobi","Kenya",-1.2921,36.8219),
    ("Lagos","Nigeria",6.5244,3.3792),("Cape Town","South Africa",-33.9249,18.4241),
    ("London","UK",51.5074,-0.1278),("Paris","France",48.8566,2.3522),
    ("Moscow","Russia",55.7558,37.6173),("Reykjavik","Iceland",64.1466,-21.9426),
    ("New York","USA",40.7128,-74.0060),("Toronto","Canada",43.6532,-79.3832),
    ("Mexico City","Mexico",19.4326,-99.1332),("Sao Paulo","Brazil",-23.5505,-46.6333),
], columns=["city","country","latitude","longitude"])

cities.to_csv(PROCESSED / "city_reference.csv", index=False)
cities

,city,country,latitude,longitude
0,Lahore,Pakistan,31.5497,74.3436
1,Karachi,Pakistan,24.8607,67.0011
2,Dubai,UAE,25.2048,55.2708
3,Delhi,India,28.6139,77.2090
4,Tokyo,Japan,35.6762,139.6503
5,Singapore,Singapore,1.3521,103.8198
6,Beijing,China,39.9042,116.4074
7,Sydney,Australia,-33.8688,151.2093
8,Cairo,Egypt,30.0444,31.2357
9,Nairobi,Kenya,-1.2921,36.8219


Choosing cities with different weather conditions (a mix of tropical, desert, temperate, and polar climates across six continents)

In [3]:
import requests

URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY = ["weather_code","temperature_2m_max","temperature_2m_min","temperature_2m_mean",
         "precipitation_sum","rain_sum","precipitation_hours",
         "wind_speed_10m_max","wind_gusts_10m_max","wind_direction_10m_dominant",
         "shortwave_radiation_sum","sunshine_duration","daylight_duration"]

test = cities.iloc[0]
params = {"latitude": test.latitude, "longitude": test.longitude,
          "start_date": "2016-01-01", "end_date": "2025-12-31",
          "daily": ",".join(DAILY), "timezone": "auto"}

r = requests.get(URL , params=params ,timeout=60)
print("Status:", r.status_code)
data = r.json()
if r.status_code != 200:
    print(data)
else:
    print(data.keys())
    print(data["daily_units"])
    print(len(data["daily"]["time"]), "days")

Status: 200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
{'time': 'iso8601', 'weather_code': 'wmo code', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'rain_sum': 'mm', 'precipitation_hours': 'h', 'wind_speed_10m_max': 'km/h', 'wind_gusts_10m_max': 'km/h', 'wind_direction_10m_dominant': '°', 'shortwave_radiation_sum': 'MJ/m²', 'sunshine_duration': 's', 'daylight_duration': 's'}
3653 days


In [7]:
import time

extra_candidates = ["relative_humidity_2m_mean", "dew_point_2m_mean",
                    "cloud_cover_mean", "pressure_msl_mean", "surface_pressure_mean"]

accepted = []
for var in extra_candidates:
    p = {"latitude": 31.5497, "longitude": 74.3436,
         "start_date": "2016-01-01", "end_date": "2016-01-10",
         "daily": var, "timezone": "auto"}
    resp = requests.get(URL, params=p, timeout=60)
    ok = resp.status_code == 200
    print(f"{var:30s} -> {'OK' if ok else 'REJECTED: ' + str(resp.json().get('reason'))}")
    if ok:
        accepted.append(var)
    time.sleep(1)

print("\nAccepted extras:", accepted)

relative_humidity_2m_mean      -> OK
dew_point_2m_mean              -> OK
cloud_cover_mean               -> OK
pressure_msl_mean              -> OK
surface_pressure_mean          -> OK

Accepted extras: ['relative_humidity_2m_mean', 'dew_point_2m_mean', 'cloud_cover_mean', 'pressure_msl_mean', 'surface_pressure_mean']


## Logged data collection run

In [8]:
import hashlib, json, time, requests
import pandas as pd
from datetime import datetime, timezone
DAILY_ALL = DAILY + accepted
TEAM_ROLL_NUMBERS = ["25L-0015", "25L-0032"]

RAW = BASE / "data" / "raw_logged"
RAW.mkdir(parents=True, exist_ok=True)
LOG_FILE = RAW / "collection_log.csv"

def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

This cell prepares the logged collection run. It stores our team roll numbers, sets the folder for the raw files, and defines a function that returns the current UTC time.

## Collection Loop

In [10]:
request_log = pd.read_csv(LOG_FILE).to_dict("records") if LOG_FILE.exists() else []
done = {r["city"] for r in request_log}

for row in cities.itertuples():
    if row.city in done:
        continue
    params = {"latitude": row.latitude, "longitude": row.longitude,
              "start_date": "2016-01-01", "end_date": "2025-12-31",
              "daily": ",".join(DAILY_ALL), "timezone": "auto"}
    requested_at = now_utc()
    requests_made, data = 0, None
    for attempt in range(5):
        requests_made += 1
        r = requests.get(URL, params=params, timeout=120)
        if r.status_code == 200:
            data = r.json()
            break
        if r.status_code == 429:
            time.sleep(65)
        else:
            print(row.city, "error:", r.status_code, r.text[:200])
            break
    if data is None:
        print(row.city, ": FAILED, not logged")
        continue

    collected_at = now_utc()
    name = row.city.replace(" ", "_")
    (RAW / f"{name}.json").write_text(json.dumps(data))

    df_city = pd.DataFrame(data["daily"]).rename(columns={"time": "date"})
    df_city.insert(0, "city", row.city)
    df_city.insert(1, "country", row.country)
    df_city.insert(2, "latitude", row.latitude)
    df_city.insert(3, "longitude", row.longitude)
    df_city["collected_at"] = collected_at
    df_city.to_csv(RAW / f"{name}.csv", index=False)

    request_log.append({"city": row.city, "requested_at": requested_at,
                        "collected_at": collected_at, "requests_made": requests_made,
                        "rows": len(df_city)})
    pd.DataFrame(request_log).to_csv(LOG_FILE, index=False)
    print(f"{row.city}: {len(df_city)} rows")
    time.sleep(3)

Lagos: 3653 rows
Cape Town: 3653 rows


This loop requests 10 years of daily data (2016 to 2025) for each city from the Open-Meteo archive API . For every city it saves the raw JSON response, adds a `collected_at` timestamp to each row, and writes the city's CSV and a line in the collection log.

All 20 cities returned 3,653 rows each. Lagos and Cape Town failed on the first pass probably because of rate limiting.

## Combining , fingerprints and printing the mandatory log

In [12]:
weather_raw = pd.concat(
    [pd.read_csv(RAW / f"{c.replace(' ', '_')}.csv") for c in cities["city"]],
    ignore_index=True)
RAW_CSV = PROCESSED / "weather_combined.csv"
weather_raw.to_csv(RAW_CSV, index=False)

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

log = pd.read_csv(LOG_FILE)
print("=== DATA COLLECTION LOG ===")
print("Team roll numbers :", ", ".join(TEAM_ROLL_NUMBERS))
print("Source / endpoint :", URL)
print("Collection start  :", log["requested_at"].min(), "(UTC)")
print("Collection end    :", log["collected_at"].max(), "(UTC)")
print("API requests made :", int(log["requests_made"].sum()))
print("Rows collected    :", f"{len(weather_raw):,}")
print("collected_at added:", "collected_at" in weather_raw.columns)
print("SHA-256 of", RAW_CSV.name, ":", sha256_of(RAW_CSV))

=== DATA COLLECTION LOG ===
Team roll numbers : 25L-0015, 25L-0032
Source / endpoint : https://archive-api.open-meteo.com/v1/archive
Collection start  : 2026-11-22T14:39:50+00:00 (UTC)
Collection end    : 2026-11-22T15:07:13+00:00 (UTC)
API requests made : 31
Rows collected    : 73,060
collected_at added: True
SHA-256 of weather_combined.csv : 4cb30dc95f0b64192a1952589d81a3e468dd7d5b90192a5117a0faad3be0a1c8


This cell joins the 20 city files into one dataset and saves it as `weather_combined.csv`. It then computes a SHA-256 fingerprint of the file.
The request count of 31 includes the retries for the cities that were logged.So the real number of requests was slightly higher.

### Data dictionary

| Column | Meaning | Unit | Type |
|---|---|---|---|
| city, country | Location names | - | Categorical |
| latitude, longitude | Coordinates | degrees | Numerical |
| date | Calendar day (local time) | - | Datetime |
| weather_code | WMO weather condition code | code | Categorical (numeric code) |
| temperature_2m_max / min / mean | Daily max/min/mean temperature at 2 m | °C | Numerical |
| precipitation_sum | Total daily precipitation | mm | Numerical |
| rain_sum | Daily rain only (excludes snow) | mm | Numerical |
| precipitation_hours | Hours with precipitation | h | Numerical |
| wind_speed_10m_max | Max wind speed at 10 m | km/h | Numerical |
| wind_gusts_10m_max | Max wind gusts at 10 m | km/h | Numerical |
| wind_direction_10m_dominant | Dominant wind direction | ° (0-360) | Numerical (circular) |
| shortwave_radiation_sum | Total solar radiation | MJ/m² | Numerical |
| sunshine_duration | Seconds of sunshine | s | Numerical |
| daylight_duration | Seconds of daylight | s | Numerical |
| collected_at | UTC time when the city's data was retrieved | – | Datetime (provenance, not a weather feature) |

**Data source note:** Data comes from the Open-Meteo Historical Weather API, which uses reanalysis datasets. These combine observations with numerical weather models, so values are estimates for a grid location and are not identical to measurements from a weather station. Attribution: Open-Meteo (https://open-meteo.com).